# 12 Building MultiStep Task Chains

Notebook generado a partir del paquete Python. Los módulos se incluyen en orden de dependencia (los módulos importados por otros aparecen primero).

> Nota: los `import` de módulos locales del paquete original se conservan tal cual. Como todos los módulos están combinados en este notebook en orden de dependencia, los símbolos referenciados ya quedan definidos en celdas anteriores.

## ¿Qué hace este notebook?

Construye una **cadena de tareas de varios pasos** con revisión humana. El flujo es
`plan → execute → review → revise`: un LLM planificador (`ChatAnthropic`) descompone la
tarea en 3 pasos, un ejecutor produce un borrador conciso, y el nodo `review` usa
`interrupt()` para que la persona apruebe, pida revisión o cambie el enfoque antes de
finalizar.

> Requiere `ANTHROPIC_API_KEY` en el entorno (archivo `.env`).

## Ejemplo de uso

**Datos de interacción que espera el agente (revisión del borrador).** El grafo se **pausa**
en `review`; necesita tu decisión para producir la salida final.

- Entrada inicial: `task` (str) es el dato relevante; `plan`/`draft`/`final` en `""`,
  `messages`/`history` en `[]`, `feedback` en `{}`.
- En la pausa el agente entrega: `{"message", "draft", "options"}`.
- Para **continuar**, `Command(resume=decision)` donde `decision` espera:
  - `action`: `"approve"` (usa el borrador tal cual como salida final) o cualquier otra
    (p. ej. `"revise"` / `"change focus"`) que dispara una revisión con el LLM
  - si NO es `"approve"`: incluye `notes` (str) con las instrucciones de revisión.

> Requiere `ANTHROPIC_API_KEY` en `.env`.

```python
from langgraph.types import Command

app = create_graph()
config = {"configurable": {"thread_id": "task-chain-demo"}}
state = {
    "messages": [], "history": [],
    "task": "Explain how API rate limiting works for beginners",
    "plan": "", "draft": "", "final": "", "feedback": {},
}

result = app.invoke(state, config=config)          # se detiene en review
print(result["__interrupt__"][0].value["draft"])    # borrador a revisar

# Aprobar tal cual:
final = app.invoke(Command(resume={"action": "approve"}), config=config)
print(final["final"])

# Alternativa (revisar con feedback):
# app.invoke(Command(resume={"action": "revise", "notes": "Hazlo más breve"}), config=config)
```

In [1]:
from __future__ import annotations

from typing import TypedDict, Annotated, Sequence, Dict, Any
import operator
import os

from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

from langchain_anthropic import ChatAnthropic

In [2]:


load_dotenv()

True

In [3]:
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY not set")

In [4]:


def build_llm():
    return ChatAnthropic(
        model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
        temperature=0.3,
    )

In [5]:


class TaskState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]
    history: Annotated[Sequence[str], operator.add]

    task: str
    plan: str
    draft: str
    final: str

    feedback: Dict[str, Any]

In [6]:

def planner_node(state: TaskState) -> TaskState:
    llm = build_llm()

    prompt = f"""
You are a task planner.

Break the following task into 3 clear steps.
Do not execute the task.

Task:
{state["task"]}
""".strip()

    plan = llm.invoke(prompt).content

    return {
        "plan": plan,
        "messages": ["Planner: task decomposed"],
        "history": ["planner_node"],
    }

In [7]:


def executor_node(state: TaskState) -> TaskState:
    llm = build_llm()

    prompt = f"""
You are an executor.

Follow this plan and produce a concise draft (max 150 words).

Plan:
{state["plan"]}

Task:
{state["task"]}
""".strip()

    draft = llm.invoke(prompt).content

    return {
        "draft": draft,
        "messages": ["Executor: draft created"],
        "history": ["executor_node"],
    }

In [8]:


def human_review_node(state: TaskState) -> TaskState:
    payload = {
        "message": "Review the draft and optionally change direction",
        "draft": state["draft"],
        "options": [
            "approve",
            "revise with feedback",
            "change focus",
        ],
    }

    decision = interrupt(payload)

    return {
        "feedback": decision,
        "messages": [f"Human review: {decision.get('action')}"],
        "history": ["human_review_node"],
    }

In [9]:


def revise_node(state: TaskState) -> TaskState:
    llm = build_llm()
    feedback = state["feedback"]

    if feedback.get("action") == "approve":
        return {
            "final": state["draft"],
            "messages": ["Executor: approved without changes"],
            "history": ["revise_node"],
        }

    prompt = f"""
Revise the draft based on human feedback.

Draft:
{state["draft"]}

Feedback:
{feedback.get("notes", "No notes")}
""".strip()

    revised = llm.invoke(prompt).content

    return {
        "final": revised,
        "messages": ["Executor: draft revised"],
        "history": ["revise_node"],
    }

In [10]:

def create_graph():
    graph = StateGraph(TaskState)

    graph.add_node("plan", planner_node)
    graph.add_node("execute", executor_node)
    graph.add_node("review", human_review_node)
    graph.add_node("revise", revise_node)

    graph.add_edge("plan", "execute")
    graph.add_edge("execute", "review")
    graph.add_edge("review", "revise")
    graph.add_edge("revise", END)

    graph.set_entry_point("plan")

    memory = InMemorySaver()
    return graph.compile(checkpointer=memory)

In [11]:


def run_demo():
    app = create_graph()

    initial_state: TaskState = {
        "messages": [],
        "history": [],
        "task": "Explain how API rate limiting works for beginners",
        "plan": "",
        "draft": "",
        "final": "",
        "feedback": {},
    }

    result = app.invoke(
        initial_state,
        config={"configurable": {"thread_id": "task-chain-demo"}},
    )

    interrupt_obj = result["__interrupt__"][0]
    payload = interrupt_obj.value

    print("\n--- HUMAN REVIEW ---")
    print(payload["draft"])

    print("\nChoose action:")
    print("1) approve")
    print("2) revise")
    print("3) change focus")

    choice = input("> ").strip()

    if choice == "1":
        decision = {"action": "approve"}
    elif choice == "2":
        notes = input("Revision notes: ")
        decision = {"action": "revise", "notes": notes}
    else:
        notes = input("New focus: ")
        decision = {"action": "change focus", "notes": notes}

    final = app.invoke(
        Command(resume=decision),
        config={"configurable": {"thread_id": "task-chain-demo"}},
    )

    print("\n--- FINAL OUTPUT ---\n")
    print(final["final"])

    print("\n--- EXECUTION TRACE ---")
    print(" -> ".join(final["history"]))

In [12]:


if __name__ == "__main__":
    run_demo()


--- HUMAN REVIEW ---
# API Rate Limiting: A Beginner's Guide

**What is it?**
An API lets apps talk to each other. Rate limiting controls *how often* you can make requests — protecting servers from overload and ensuring fair access for everyone.

**How it works:**
Limits are measured as requests per second, minute, or hour. Common methods include:
- **Fixed window** – 100 requests per hour, reset on the clock
- **Token bucket** – earn tokens over time, spend one per request
- **Sliding window** – a rolling time frame tracking recent requests

Exceed your limit and you'll get a `429 Too Many Requests` error.

**Real-world analogy:**
Think of a coffee shop with one barista — they can only serve so many customers per hour. Too many orders? You wait.

**Tips for your code:**
- Check response headers for limit info (`X-RateLimit-Remaining`)
- Add delays between requests
- Cache responses to avoid repeat calls
- Retry automatically after waiting

Choose action:
1) approve
2) revise
3) chang